In [1]:
import emodeconnection as emc
import numpy as np
import matplotlib.pyplot as plt
import time

## --- 1. Sweep Parameters ---
# Sweep thickness from 300nm to 800nm (adjust based on your growth capability)
thicknesses = np.arange(300, 400, 10)
neff_fund = []
neff_shg = []

# Fixed parameters
wavelength_fund = 450.0
w_core = 400.0  # [nm] Fixed width
w_trench = 1000.0  # [nm] Trench width on either side of the core
window_width = w_core + w_trench * 2
dx, dy = 10.0, 10.0
target_mode_shg = None  # We will define this via prompt on the first run

em = emc.EMode()

# Sellmeier Material Definition
# Use double asterisks for power, ensure no spaces in the inner math, 
# and verify the negative B3/C3 values are handled cleanly.
eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"
em.add_material(name='custom_AlN', refractive_index_equation=anisotropic_equation, wavelength_unit='um')

print(f"{'Thick (nm)':<12} | {'n_eff 450':<12} | {'n_eff 225':<12} | {'Delta N'}")
print("-" * 60)
 


Thick (nm)   | n_eff 450    | n_eff 225    | Delta N
------------------------------------------------------------


In [3]:
h = 350
# --- Solve Fundamental (450nm) ---
em.settings(wavelength=wavelength_fund, x_resolution=dx, y_resolution=dy,
            window_width=2000, window_height=h+2000, num_modes=3)
    
em.shape(name='Substrate', material='Al2O3', height=1000)
em.shape(name='core', material='custom_AlN', height=h, mask=w_core, sidewall_angle=5)
em.shape(name='TopClad', material='SiO2', height=800, shape_type='conformal')
    

# Check if modes actually exist before calling get_neff
em.FDM()

{'field': Field(field=array([[[[-1.68115324e-18-0.j, -1.66328895e-18-0.j,
           -1.37126446e-18-0.j, ..., -1.59278593e-18-0.j,
           -1.97771429e-18-0.j, -1.31676432e-18-0.j],
          [-2.46557473e-18-0.j, -2.21998044e-18-0.j,
           -1.63411235e-18-0.j, ..., -1.96216763e-18-0.j,
           -1.03590112e-18-0.j, -6.61145102e-19-0.j],
          [-3.22723295e-18-0.j, -3.05256777e-18-0.j,
           -2.44920845e-18-0.j, ..., -1.19772898e-18-0.j,
           -9.91712076e-19-0.j, -2.37446650e-19-0.j],
          ...,
          [ 1.88250661e-18-0.j,  2.46318420e-18-0.j,
            1.58749643e-18-0.j, ...,  3.60162374e-21-0.j,
            2.63176864e-20-0.j,  2.73209484e-19-0.j],
          [ 1.79829128e-18-0.j,  2.13590700e-18-0.j,
            1.94531541e-18-0.j, ..., -2.59472690e-19-0.j,
            3.69178293e-20-0.j,  3.01172800e-19-0.j],
          [ 1.77715609e-18-0.j,  2.36129004e-18-0.j,
            1.95301160e-18-0.j, ..., -2.33866486e-19-0.j,
           -3.60223843e-19-0

In [4]:
em.plot()

try:
    data = em.get_report()
    n450 = data['neff'][0] # Index 0 is the first mode
    print(f"Success: n_eff = {n450}")
    
except KeyError:
    print("Error: 'neff' not found in report. Did the solver converge?")
    n450 = np.nan
except Exception as e:
    print(f"Unexpected connection error: {e}")
    n450 = np.nan

neff_fund.append(n450)



Unexpected connection error: 'EM_get_report'


In [32]:
import emodeconnection as emc
import numpy as np

## Set simulation parameters
wavelength = 450 # [nm] wavelength
dx, dy = 10, 10 # [nm] resolution
w_core = 400 # [nm] waveguide core width
w_trench = 800 # [nm] waveguide side trench width
h_core = 340 # [nm] waveguide core height
h_clad_top = 800 # [nm] waveguide top and bottom clad
h_clad_bottom = 1000 # [nm] waveguide bottom clad
window_width = w_core + w_trench*2 # [nm]
window_height = h_core + h_clad_top + h_clad_bottom # [nm]
num_modes = 10 # [-] number of modes
boundary = 'TM' # boundary condition  or '00' or '0S'

## Connect and initialize EMode
em = emc.EMode()

## Settings
em.settings(
    wavelength = wavelength, x_resolution = dx, y_resolution = dy,
    window_width = window_width, window_height = window_height,
    num_modes = num_modes, background_material = 'Air',
    boundary_condition = boundary,
)


'ok'

In [18]:
# see the SiN example for how to write a Sellmeier equation
#        equation = '(1 + 0.6961663/(1 - (0.0684043/x)^2) + 0.4079426/(1 - (0.1162414/x)^2) + 0.8974794/(1 - (9.896161/x)^2))^0.5'
# Coeffs: [B1, C1, B2, C2, B3, C3]
# n_o: [2.8032 0.015287 0.36335 0.036095 -3.3508e+07 -3.672e+08]
# n_e: [0.017061 0.043855 3.1976 0.022642 -5.7269e+07 7.4226e+07]
#equation_o = '(1 + 2.8032/(1 - (0.015287/x)^2) + 0.36335/(1 - (0.036095/x)^2) + -3.3508e+07/(1 - (-3.672e+08/x)^2))^0.5'
#equation_e = '(1 + 0.017061/(1 - (0.043855/x)^2) + 3.1976/(1 - (0.022642/x)^2) + -5.7269e+07/(1 - (7.4226e+07/x)^2))^0.5'
equation_o = '(1 + 2.8032/(1 - 0.015287/x^2) + 0.36335/(1 - 0.036095/x^2) + -33508000/(1 - (-367200000)/x^2))^0.5'
equation_e = '(1 + 0.017061/(1 - 0.043855/x^2) + 3.1976/(1 - 0.022642/x^2) + -57269000/(1 - 74226000/x^2))^0.5'
anisotropic_equation = f"[{equation_o}, {equation_e}, {equation_o}]"


In [19]:

em.add_material(name = 'custom_AlN',
    refractive_index_equation = anisotropic_equation, wavelength_unit = 'um')


## View material database
em.material_explorer()

'ok'

In [20]:
 ## Draw shapes
em.shape(name = 'Substrate', material = 'Al2O3', height = h_clad_bottom)
em.shape(name = 'core', material = 'custom_AlN', height = h_core,
    etch_depth = h_core*1, mask = w_core, sidewall_angle = 15)
em.shape(name = 'TopClad', material = 'SiO2', height = h_clad_top, shape_type='conformal')
em.plot()

'ok'

In [21]:
em.FDM()                # Solve for the modes  (as many as was request num_modes) 


{'field': {'data': array([[[[ 2.12971050e-12+0.j,  2.15487202e-12+0.j,
             2.20542572e-12+0.j, ..., -4.73533598e-15+0.j,
            -3.07640691e-15+0.j, -1.51480995e-15+0.j],
           [ 4.27430926e-12+0.j,  4.32487347e-12+0.j,
             4.42646773e-12+0.j, ..., -7.88546798e-15+0.j,
            -5.12306495e-15+0.j, -2.52205503e-15+0.j],
           [ 6.44872140e-12+0.j,  6.52517258e-12+0.j,
             6.67878363e-12+0.j, ..., -1.12681443e-14+0.j,
            -7.32157933e-15+0.j, -3.60411783e-15+0.j],
           ...,
           [-2.39437836e-11+0.j, -2.41950134e-11+0.j,
            -2.46994326e-11+0.j, ...,  2.31277706e-14+0.j,
             1.50171064e-14+0.j,  7.38912251e-15+0.j],
           [-2.35506487e-11+0.j, -2.37964551e-11+0.j,
            -2.42899673e-11+0.j, ...,  2.18897952e-14+0.j,
             1.42120381e-14+0.j,  6.99266595e-15+0.j],
           [-2.33545740e-11+0.j, -2.35976792e-11+0.j,
            -2.40857574e-11+0.j, ...,  2.12802893e-14+0.j,
             1

In [22]:
## Actually run the eMode solver .... should print out em.report()  in terminal 
em.confinement()        # Calculate the confinement factor for each mode  (% of mode in a given shape,  including 'background')

{'TopClad': {'0': 0.03447612994521545,
  '1': 0.034686097252058366,
  '2': 0.06659701569210479,
  '3': 0.1182192126254088,
  '4': 0.04705356448818406,
  '5': 0.015870541422088283,
  '6': 0.004598285308430278,
  '7': 0.0025333884843990353,
  '8': 0.0025577273526639433,
  '9': 0.0014922683721225844},
 'core': {'0': 0.9306623269621518,
  '1': 0.9145799575302827,
  '2': 0.6640663290123103,
  '3': 0.7708697051178541,
  '4': 0.4162687346329501,
  '5': 0.14254052717889196,
  '6': 0.03761256112295831,
  '7': 0.009567190903287393,
  '8': 0.010517166386634311,
  '9': 0.0035685519737443843},
 'Substrate': {'0': 0.0348615430926328,
  '1': 0.050733945217658775,
  '2': 0.26933665529526013,
  '3': 0.11091108225663888,
  '4': 0.5366777008785258,
  '5': 0.8415889313988079,
  '6': 0.957789153568544,
  '7': 0.9878994206121854,
  '8': 0.9869251062606088,
  '9': 0.9949391796541046},
 'background': {'0': 5.160792526950013e-17,
  '1': 4.1212195612867593e-16,
  '2': 3.248077947373202e-13,
  '3': 9.79449553007

In [23]:
em.scattering()         # generates loss estimates based on the geometry and assumed scattering parameters (roughness, correlation length, etc.)  see "Laser.example" on emode webbsite



array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [31]:
em.report()
em.label_profile(name = '0') # save the data from the first mode
## Plot the field and refractive index profiles
em.plot()

'ok'

In [27]:
## Calculate the overlap integral
#overlap = em.overlap(label_a = '0', mode_a = 0,
#    label_b = '1', mode_b = 0)
#print('Power overlap: %0.3f %%' % (overlap*100))

In [29]:
## Display scattering loss details from the core
shape_core = em.get('shape_core')
sv = shape_core['metadata']



In [30]:
sh = shape_core['metadata']['scattering_horizontal_edges']
sT = shape_core['metadata']['scattering_sum']
print('Scattering loss from all vertical edges: %0.1f dB/m' % sv[0])
print('Scattering loss from all horizontal edges: %0.1f dB/m' % sh[0])
print('Total scattering loss: %0.1f dB/m\n' % sT[0])

KeyError: 'scattering_horizontal_edges'

In [ ]:
## INCREASE THE WIDTH + Solve the second set of modes   
em.shape(name = 'core', mask = w_core + 100)    #this updates the geometry
em.FDM()
em.confinement()
em.scattering()
em.report()
em.label_profile(name = '1') # save the data from the first mode

## Calculate the overlap integral
overlap = em.overlap(label_a = '0', mode_a = 0,
    label_b = '1', mode_b = 0)
print('Power overlap: %0.3f %%' % (overlap*100))

## Plot the fields
em.plot()


## Get the refractive index of the core
n_ALN = em.refractive_index(material = 'custom_AlN',
                          wavelength = wavelength)
print('\nRefractive index of ALN:', n_ALN)

## Close EMode
em.close()


Power overlap: 98.357 %


In [5]:
em.close()